In [1]:
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns
import re

In [2]:
csv_file = Path("../data/customer_support_tickets.csv")
csv_f = "../data/customer_support_tickets.csv"
print(csv_file)
print(csv_f)

df = pd.read_csv(csv_file)
df.head()

..\data\customer_support_tickets.csv
../data/customer_support_tickets.csv


,Ticket ID,Customer Name,Customer Email,Customer Age,Customer Gender,Product Purchased,Date of Purchase,Ticket Type,Ticket Subject,Ticket Description,Ticket Status,Resolution,Ticket Priority,Ticket Channel,First Response Time,Time to Resolution,Customer Satisfaction Rating
0,1,Marisa Obrien,carrollallison@example.com,32,Other,GoPro Hero,2021-03-22,Technical issue,Product setup,I'm having an issue with the {product_purchase...,Pending Customer Response,NaN,Critical,Social media,2023-06-01 12:15:36,NaN,NaN
1,2,Jessica Rios,clarkeashley@example.com,42,Female,LG Smart TV,2021-05-22,Technical issue,Peripheral compatibility,I'm having an issue with the {product_purchase...,Pending Customer Response,NaN,Critical,Chat,2023-06-01 16:45:38,NaN,NaN
2,3,Christopher Robbins,gonzalestracy@example.com,48,Other,Dell XPS,2020-07-14,Technical issue,Network problem,I'm facing a problem with my {product_purchase...,Closed,Case maybe show recently my computer follow.,Low,Social media,2023-06-01 11:14:38,2023-06-01 18:05:38,3.0
3,4,Christina Dillon,bradleyolson@example.org,27,Female,Microsoft Office,2020-11-13,Billing inquiry,Account access,I'm having an issue with the {product_purchase...,Closed,Try capital clearly never color toward story.,Low,Social media,2023-06-01 07:29:40,2023-06-01 01:57:40,3.0
4,5,Alexander Carroll,bradleymark@example.com,67,Female,Autodesk AutoCAD,2020-02-04,Billing inquiry,Data loss,I'm having an issue with the {product_purchase...,Closed,West decision evidence bit.,Low,Email,2023-06-01 00:12:42,2023-06-01 19:53:42,1.0


In [3]:
pd.set_option('display.max_colwidth', None)

#### Embedding preprocess

In [5]:
df['Ticket_Description'] = df.apply(lambda row: str(row['Ticket Description']).replace('{product_purchased}', str(row['Product Purchased'])), axis=1)

print(df[['Ticket Description', 'Ticket_Description']].head())

                                                                                                                                                                                                                                                                                                                                       Ticket Description  \
0                                          I'm having an issue with the {product_purchased}. Please assist.\r\n\r\nYour billing zip code is: 71701.\r\n\r\nWe appreciate that you have requested a website address.\r\n\r\nPlease double check your email address. I've tried troubleshooting steps mentioned in the user manual, but the issue persists.   
1                                            I'm having an issue with the {product_purchased}. Please assist.\r\n\r\nIf you need to change an existing product.\r\n\r\nI'm having an issue with the {product_purchased}. Please assist.\r\n\r\nIf The issue I'm facing is intermittent. Sometimes it works fin

In [4]:
def cleaning_descrn(row):
    text = str(row['Ticket Description']).lower().strip()
    
    #find anything inside {{ }} or { } and replace it with the Product Name
    product = str(row['Product Purchased'])
    text = re.sub(r'\{+.*?product_?purchased.*?\}+', product, text)
    
    # Since we don't have a column for 'error_message' - doubt
    # text = re.sub(r'\{+.*?error_message.*?\}+', '?', text)
    
    # Removes html tags
    text = re.sub(r'<[^>]+>', ' ', text)
    
    # Removes \r, \n, #, _, and extra brackets - } ]
    text = text.replace('\r', ' ').replace('\n', ' ')
    text = re.sub(r'[#__{}\[\]]', ' ', text)
    
    text = text.lower().strip()

    # Shrink multiple spaces to one
    text = re.sub(r'\s+', ' ', text) 
    
    return text

df['Ticket_Description'] = df.apply(cleaning_descrn, axis=1)



In [ ]:
# Check if any other curly brackets exist in the data - yet to work on it
remaining_placeholders = df['Ticket_Description'].str.contains(r'\{.*\}')
df[remaining_placeholders]['Ticket_Description']

# 27, 59, 64,65,70,8398, 8416, 8421, 8441, 8447

In [ ]:

df.loc[[27, 59, 64,65,70,8398, 8416, 8421, 8441, 8447], 'Ticket_Description'].values

In [ ]:
df.head()

In [ ]:
# word count

df['word_count'] = df['Ticket_Description'].apply(lambda x: len(str(x).split()))


In [ ]:
plt.figure(figsize=(10, 5))
sns.histplot(df['word_count'], bins=50, kde=True)
plt.title('Distribution of Word Counts')
plt.axvline(x=128, color='r', linestyle='--', label='Common Embedding Limit')
plt.legend()
plt.show()

In [ ]:
# duplicates check

df[df['Ticket_Description'].duplicated()]['Ticket_Description']

In [ ]:
df['Ticket_Description'].duplicated().sum()

In [ ]:
df['Ticket_Description'].value_counts()[df['Ticket_Description'].value_counts() > 1]

In [ ]:
df.shape

In [ ]:
import seaborn as sns

df['word_count'] = df['Ticket_Description'].apply(lambda x: len(str(x).split()))

plt.figure(figsize=(8,5))
sns.histplot(df['word_count'], bins=30, kde=True, color='orange')
plt.title('Distribution of Word Counts')
plt.show()

print(f"Average words per ticket: {df['word_count'].mean():.2f}")

In [ ]:
from wordcloud import WordCloud
import matplotlib.pyplot as plt

# Combine all text into one giant string
all_text = " ".join(df['Ticket_Description'].astype(str))

wordcloud = WordCloud(width=800, height=400, background_color='white', max_words=100).generate(all_text)

plt.figure(figsize=(10, 5))
plt.imshow(wordcloud, interpolation='bilinear')
plt.axis('off')
plt.title('Top 100 Most Frequent Words in Raw Data')
plt.show()

#### TF-IDF preprocess

In [ ]:
df.info()

In [ ]:
df['Ticket Description']

### Duplicate analysis

In [13]:
df[df['Ticket Description'].duplicated(keep=False)]


,Ticket ID,Customer Name,Customer Email,Customer Age,Customer Gender,Product Purchased,Date of Purchase,Ticket Type,Ticket Subject,Ticket Description,Ticket Status,Resolution,Ticket Priority,Ticket Channel,First Response Time,Time to Resolution,Customer Satisfaction Rating,Ticket_Description
56,57,Alison Ford,mcgeebrandon@example.org,69,Male,Autodesk AutoCAD,2021-12-22,Technical issue,Software bug,I'm having an issue with the {product_purchased}. Please assist. I've noticed that the issue occurs consistently when I use a specific feature or application on my {product_purchased}.,Pending Customer Response,NaN,Medium,Social media,2023-06-01 10:33:53,NaN,NaN,i'm having an issue with the autodesk autocad. please assist. i've noticed that the issue occurs consistently when i use a specific feature or application on my autodesk autocad.
80,81,Sarah Lee,dclark@example.net,50,Other,Microsoft Office,2021-03-22,Technical issue,Product recommendation,I'm having an issue with the {product_purchased}. Please assist. I'm unable to find the option to perform the desired action in the {product_purchased}. Could you please guide me through the steps?,Pending Customer Response,NaN,Critical,Phone,2023-06-01 07:27:28,NaN,NaN,i'm having an issue with the microsoft office. please assist. i'm unable to find the option to perform the desired action in the microsoft office. could you please guide me through the steps?
105,106,Virginia Miller,lisahamilton@example.com,56,Other,Dyson Vacuum Cleaner,2021-07-17,Product inquiry,Battery life,I'm having an issue with the {product_purchased}. Please assist. I need assistance as soon as possible because it's affecting my work and productivity.,Pending Customer Response,NaN,Critical,Chat,2023-06-01 02:08:01,NaN,NaN,i'm having an issue with the dyson vacuum cleaner. please assist. i need assistance as soon as possible because it's affecting my work and productivity.
106,107,Miranda Morales,daniellebrown@example.net,42,Female,Nintendo Switch,2021-05-20,Cancellation request,Installation support,I'm having an issue with the {product_purchased}. Please assist. I've checked the device settings and made sure that everything is configured correctly.,Closed,Table admit really Mrs development move.,Low,Email,2023-06-01 11:38:01,2023-06-01 20:46:01,2.0,i'm having an issue with the nintendo switch. please assist. i've checked the device settings and made sure that everything is configured correctly.
107,108,Laura Collins,douglasmccormick@example.net,50,Other,LG Smart TV,2021-06-09,Technical issue,Account access,"I'm having an issue with the {product_purchased}. Please assist. I've checked for software updates, and my {product_purchased} is already running the latest version.",Pending Customer Response,NaN,High,Email,2023-06-01 15:49:01,NaN,NaN,"i'm having an issue with the lg smart tv. please assist. i've checked for software updates, and my lg smart tv is already running the latest version."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8386,8387,Caitlin Robinson,bsmith@example.com,24,Female,Dyson Vacuum Cleaner,2020-02-11,Refund request,Installation support,I'm having an issue with the {product_purchased}. Please assist. I'm concerned about the security of my {product_purchased} and would like to ensure that my data is safe.,Pending Customer Response,NaN,Critical,Social media,2023-06-01 12:50:37,NaN,NaN,i'm having an issue with the dyson vacuum cleaner. please assist. i'm concerned about the security of my dyson vacuum cleaner and would like to ensure that my data is safe.
8397,8398,Richard Mccarthy,xbailey@example.com,59,Female,Dell XPS,2020-07-17,Cancellation request,Cancellation request,"I'm having an issue with the {product_purchased}. Please assist. The issue I'm facing is intermittent. Sometimes it works fine, but other times it acts up unexpectedly.",Closed,Because individual section grow.,Medium,Email,2023-06-01 06:30:51,2023-06-01 15:11:51,2.0,"i'm having an issue with the dell xps. 

In [14]:
df['Ticket Description'].nunique()

8077

In [15]:
df['Ticket Description'][0]

"I'm having an issue with the {product_purchased}. Please assist.\r\n\r\nYour billing zip code is: 71701.\r\n\r\nWe appreciate that you have requested a website address.\r\n\r\nPlease double check your email address. I've tried troubleshooting steps mentioned in the user manual, but the issue persists."

In [16]:
df.shape

(8469, 18)

In [6]:
df_dup = df[df['Ticket_Description'].duplicated(keep=False)].copy()
df_dup['row_id'] = df_dup.index

df_dup.sort_values('Ticket_Description')

,Ticket ID,Customer Name,Customer Email,Customer Age,Customer Gender,Product Purchased,Date of Purchase,Ticket Type,Ticket Subject,Ticket Description,Ticket Status,Resolution,Ticket Priority,Ticket Channel,First Response Time,Time to Resolution,Customer Satisfaction Rating,Ticket_Description,row_id
7054,7055,Kyle Mcdonald,dpatton@example.org,27,Other,Amazon Echo,2020-09-13,Product inquiry,Product compatibility,"I'm having an issue with the {product_purchased}. Please assist. I've already contacted customer support multiple times, but the issue remains unresolved.",Closed,Where weight herself give rich lead.,Low,Email,2023-06-01 11:50:31,2023-06-01 14:21:31,2.0,"i'm having an issue with the amazon echo. please assist. i've already contacted customer support multiple times, but the issue remains unresolved.",7054
5679,5680,Donald Houston,kleon@example.org,45,Female,Amazon Echo,2021-12-18,Refund request,Product compatibility,"I'm having an issue with the {product_purchased}. Please assist. I've already contacted customer support multiple times, but the issue remains unresolved.",Open,NaN,Critical,Email,NaN,NaN,NaN,"i'm having an issue with the amazon echo. please assist. i've already contacted customer support multiple times, but the issue remains unresolved.",5679
871,872,Anthony Adams,lmalone@example.com,25,Male,Amazon Echo,2020-09-10,Billing inquiry,Cancellation request,I'm having an issue with the {product_purchased}. Please assist. This problem started occurring after the recent software update. I haven't made any other changes to the device.,Pending Customer Response,NaN,Low,Email,2023-06-01 12:05:37,NaN,NaN,i'm having an issue with the amazon echo. please assist. this problem started occurring after the recent software update. i haven't made any other changes to the device.,871
8168,8169,Rebecca White,matthew09@example.org,66,Male,Amazon Echo,2021-06-06,Refund request,Battery life,I'm having an issue with the {product_purchased}. Please assist. This problem started occurring after the recent software update. I haven't made any other changes to the device.,Pending Customer Response,NaN,Critical,Email,2023-06-01 13:16:39,NaN,NaN,i'm having an issue with the amazon echo. please assist. this problem started occurring after the recent software update. i haven't made any other changes to the device.,8168
2416,2417,Mackenzie Thomas,kimberlyhamilton@example.net,66,Male,Amazon Kindle,2020-01-06,Refund request,Cancellation request,I'm having an issue with the {product_purchased}. Please assist. I've noticed a peculiar error message popping up on my {product_purchased} screen. It says '{error_message}'. What does it mean?,Closed,Them general professional gun.,Medium,Chat,2023-06-01 19:28:26,2023-06-01 09:07:26,3.0,i'm having an issue with the amazon kindle. please assist. i've noticed a peculiar error message popping up on my amazon kindle screen. it says ' error message '. what does it mean?,2416
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4751,4752,Stephanie Bowman,lisabradley@example.com,62,Female,Xbox,2020-05-27,Cancellation request,Refund request,I'm having an issue with the {product_purchased}. Please assist. I need assistance as soon as possible because it's affecting my work and productivity.,Pending Customer Response,NaN,Medium,Email,2023-06-01 05:57:49,NaN,NaN,i'm having an issue with the xbox. please assist. i need assistance as soon as possible because it's affecting my work and productivity.,4751
6115,6116,Victor Todd,nelsonroy@example.com,41,Male,Xbox,2020-09-28,Refund request,Software bug,"I'm having an issue with the {product_purchased}. Please assist. I've checked for software updates, and my {product_purchased} is already running the latest version.",Closed,Little style decade cold.,High,Phone,2023-06-01 05:10:04,2023-06-01 23:50:04,2.0,"i'm having an issue with the xbox. please assist. i've checked for software updates, and my xbox is already running the latest version.",6115
311

In [5]:
df['Ticket_Description'] = df.apply(cleaning_descrn, axis=1)

dup_groups = (
    df[df['Ticket_Description'].duplicated(keep=False)]
    .groupby('Ticket_Description')
    .apply(lambda x: x.index.tolist())
)

print(dup_groups)

Ticket_Description
i'm having an issue with the amazon echo. please assist. i've already contacted customer support multiple times, but the issue remains unresolved.                                       [5679, 7054]
i'm having an issue with the amazon echo. please assist. this problem started occurring after the recent software update. i haven't made any other changes to the device.                 [871, 8168]
i'm having an issue with the amazon kindle. please assist. i've noticed a peculiar error message popping up on my amazon kindle screen. it says ' error message '. what does it mean?    [2416, 6176]
i'm having an issue with the amazon kindle. please assist. the issue i'm facing is intermittent. sometimes it works fine, but other times it acts up unexpectedly.                       [3389, 5113]
i'm having an issue with the apple airpods. please assist. i rely heavily on my apple airpods for my daily tasks, and this issue is hindering my productivity.                           [422

In [ ]:
df.head()
df['Ticket_Description'] = df.apply(cleaning_descrn, axis=1)

In [22]:
text =  df['Ticket Description']
text.duplicated().sum()

np.int64(392)

In [12]:
df['Ticket_Description'].duplicated().sum()

np.int64(74)

In [9]:
data = df['Ticket_Description'].unique()
type(data)
len(data)

8395

In [11]:
df.shape

(8469, 18)

In [17]:
data

<StringArray>
[                                                                              'i'm having an issue with the gopro hero. please assist. your billing zip code is: 71701. we appreciate that you have requested a website address. please double check your email address. i've tried troubleshooting steps mentioned in the user manual, but the issue persists.',
                                                                                        'i'm having an issue with the lg smart tv. please assist. if you need to change an existing product. i'm having an issue with the lg smart tv. please assist. if the issue i'm facing is intermittent. sometimes it works fine, but other times it acts up unexpectedly.',
                                                                                                              'i'm facing a problem with my dell xps. the dell xps is not turning on. it was working fine until yesterday, but now it doesn't respond. 1.8.3 i really i'm using the 

In [18]:
import nltk 

nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('wordnet')

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\Saranya\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\Saranya\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\Saranya\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

In [21]:
data

<StringArray>
[                                                                              'i'm having an issue with the gopro hero. please assist. your billing zip code is: 71701. we appreciate that you have requested a website address. please double check your email address. i've tried troubleshooting steps mentioned in the user manual, but the issue persists.',
                                                                                        'i'm having an issue with the lg smart tv. please assist. if you need to change an existing product. i'm having an issue with the lg smart tv. please assist. if the issue i'm facing is intermittent. sometimes it works fine, but other times it acts up unexpectedly.',
                                                                                                              'i'm facing a problem with my dell xps. the dell xps is not turning on. it was working fine until yesterday, but now it doesn't respond. 1.8.3 i really i'm using the 

In [27]:
lemmatizer = nltk.WordNetLemmatizer()

preprocessed_data = []
for sent in data:
    tokens = nltk.word_tokenize(sent)
    temp = ""
    for token in tokens:
        if re.search(r'[^a-zA-Z0-9]', token):
            print(token)
        else:
            temp = ' '.join([temp, lemmatizer.lemmatize(lemmatizer.lemmatize(token, pos='v'), pos = 'n')])

    preprocessed_data.append(temp)


'm
.
.
:
71701.
.
.
've
,
.
'm
.
.
.
'm
.
.
'm
.
,
.
'm
.
.
,
n't
.
1.8.3
'm
,
's
.
'm
.
.
're
'd
,
.
've
,
.
'm
.
.
:
.
've
.
.
'm
.
.
,
n't
.
,
.
'm
.
'invalid
'
,
'm
.
?
'm
.
?
'm
.
.
(
)
.
,
.
've
,
,
n't
.
'm
.
.
.
,
.
've
,
.
?
.
.
?
,
``
''
'm
.
,
.
'm
.
.
1-800-799-0808.
:
's
2-3-4-5
?
'm
,
's
.
'm
.
.
4.
mr.
.
5.
've
,
n't
.
'm
.
.
:
n't
?
.
:
's
've
.
've
.
've
,
.
?
n't
'product
'
'm
.
'm
.
.
:
:
:
11,532
:
've
,
.
?
'm
.
.
,
,
've
.
.
'm
.
.
-
:
.
,
'm
.
'm
.
.
!
''
*
-
-
-
've
,
n't
.
'm
.
.
...
'm
.
,
.
'm
.
.
'm
.
.
:
've
,
.
'm
.
assist.
``
-name
``
pro.
``
``
-version
1.10.2
``
1.10.2
''
``
-usage
've
,
.
'm
.
.
:
@
julietr.com
,
.
've
,
.
'm
.
.
(
,
.
)
1.3.2.1
3.0
.
'm
.
,
.
'm
.
.
'll
.
've
,
,
n't
.
'm
.
.
.
,
've
.
.
.
.
?
,
.
'm
,
.
'm
.
,
.
?
,
.
've
,
,
n't
.
'm
.
.
.
'm
.
,
.
'm
.
.
.
''
.
``
'm
,
's
.
'm
.
.
's
.
've
,
.
'm
.
.
'm
.
.
've
.
?
've
.
'm
.
.
$
29.89
.
,
've
,
.
've
.
's
.
?
's
.
'm
.
.
:
``
-
,
,
'm
.
'm
.
.
3-3
,
,
.
.
3-2
've
,
.
?
've
,
.
?
,

In [28]:
preprocessed_data

[' i have an issue with the gopro hero please assist your bill zip code be we appreciate that you have request a website address please double check your email address i try troubleshoot step mention in the user manual but the issue persist',
 ' i have an issue with the lg smart tv please assist if you need to change an exist product i have an issue with the lg smart tv please assist if the issue i face be intermittent sometimes it work fine but other time it act up unexpectedly',
 ' i face a problem with my dell xps the dell xps be not turn on it be work fine until yesterday but now it do respond i really i use the original charger that come with my dell xps but it not charge properly',
 ' i have an issue with the microsoft office please assist if you have a problem you interest in and i love to see this happen please check out the feedback i already contact customer support multiple time but the issue remain unresolved',
 ' i have an issue with the autodesk autocad please assist note

In [30]:
from sklearn.feature_extraction.text import CountVectorizer

vectorizer = CountVectorizer()
vectorizer.fit(preprocessed_data)

,"input input: {'filename', 'file', 'content'}, default='content'- If `'filename'`, the sequence passed as an argument to fit is expected to be a list of filenames that need reading to fetch the raw content to analyze.- If `'file'`, the sequence items must have a 'read' method (file-like object) that is called to fetch the bytes in memory.- If `'content'`, the input is expected to be a sequence of items that can be of type string or byte.",'content'
,"encoding encoding: str, default='utf-8'If bytes or files are given to analyze, this encoding is used todecode.",'utf-8'
,"decode_error decode_error: {'strict', 'ignore', 'replace'}, default='strict'Instruction on what to do if a byte sequence is given to analyze thatcontains characters not of the given `encoding`. By default, it is'strict', meaning that a UnicodeDecodeError will be raised. Othervalues are 'ignore' and 'replace'.",'strict'
,"strip_accents strip_accents: {'ascii', 'unicode'} or callable, default=NoneRemove accents and perform other character normalizationduring the preprocessing step.'ascii' is a fast method that only works on characters that havea direct ASCII mapping.'unicode' is a slightly slower method that works on any characters.None (default) means no character normalization is performed.Both 'ascii' and 'unicode' use NFKD normalization from:func:`unicodedata.normalize`.",None
,"lowercase lowercase: bool, default=TrueConvert all characters to lowercase before tokenizing.",True
,"preprocessor preprocessor: callable, default=NoneOverride the preprocessing (strip_accents and lowercase) stage whilepreserving the tokenizing and n-grams generation steps.Only applies if ``analyzer`` is not callable.",None
,"tokenizer tokenizer: callable, default=NoneOverride the string tokenization step while preserving thepreprocessing and n-grams generation steps.Only applies if ``analyzer == 'word'``.",None
,"stop_words stop_words: {'english'}, list, default=NoneIf 'english', a built-in stop word list for English is used.There are several known issues with 'english' and you shouldconsider an alternative (see :ref:`stop_words`).If a list, that list is assumed to contain stop words, all of whichwill be removed from the resulting tokens.Only applies if ``analyzer == 'word'``.If None, no stop words will be used. In this case, setting `max_df`to a higher value, such as in the range (0.7, 1.0), can automatically detectand filter stop words based on intra corpus document frequency of terms.",None
,"token_pattern token_pattern: str or None, default=r""(?u)\\b\\w\\w+\\b""Regular expression denoting what constitutes a ""token"", only usedif ``analyzer == 'word'``. The default regexp select tokens of 2or more alphanumeric characters (punctuation is completely ignoredand always treated as a token separator).If there is a capturing group in token_pattern then thecaptured group content, not the entire match, becomes the token.At most one capturing group is permitted.",'(?u)\\b\\w\\w+\\b'
,"ngram_range ngram_range: tuple (min_n, max_n), default=(1, 1)The lower and upper boundary of the range of n-values for differentword n-grams or char n-grams to be extracted. All values of n suchsuch that min_n <= n <= max_n will be used. For example an``ngram_range`` of ``(1, 1)`` means only unigrams, ``(1, 2)`` meansunigrams and bigrams, and ``(2, 2)`` means only bigrams.Only applies if ``analyzer`` is not callable.","(1, ...)"
,"analyzer analyzer: {'word', 'char', 'char_wb'} or callable, default='word'Whether the feature should be made of word n-gram or charactern-grams.Option 'char_wb' creates character n-grams only from text insideword boundaries; n-grams at the edges of words are padded with space.If a callable is passed it is used to extract the sequence of featuresout of the raw, unprocessed input... versionchanged:: 0.21Since v0.21, if ``input`` is ``filename`` or ``file``, the data isfirst read from the file and then passed to the given callableanalyzer.",'word'


In [31]:
vectorizer.vocabulary_

{'have': 1985,
 'an': 481,
 'issue': 2265,
 'with': 4601,
 'the': 4157,
 'gopro': 1909,
 'hero': 2016,
 'please': 3129,
 'assist': 576,
 'your': 4687,
 'bill': 727,
 'zip': 4703,
 'code': 1009,
 'be': 692,
 'we': 4531,
 'appreciate': 533,
 'that': 4152,
 'you': 4684,
 'request': 3511,
 'website': 4538,
 'address': 393,
 'double': 1418,
 'check': 939,
 'email': 1503,
 'try': 4298,
 'troubleshoot': 4290,
 'step': 3979,
 'mention': 2623,
 'in': 2136,
 'user': 4404,
 'manual': 2556,
 'but': 838,
 'persist': 3067,
 'lg': 2448,
 'smart': 3859,
 'tv': 4307,
 'if': 2106,
 'need': 2775,
 'to': 4233,
 'change': 926,
 'exist': 1605,
 'product': 3248,
 'face': 1642,
 'intermittent': 2212,
 'sometimes': 3883,
 'it': 2268,
 'work': 4615,
 'fine': 1708,
 'other': 2946,
 'time': 4220,
 'act': 368,
 'up': 4384,
 'unexpectedly': 4345,
 'problem': 3238,
 'my': 2739,
 'dell': 1274,
 'xps': 4663,
 'not': 2829,
 'turn': 4305,
 'on': 2903,
 'until': 4380,
 'yesterday': 4678,
 'now': 2840,
 'do': 1395,
 'resp

In [34]:
term_doc_mat = vectorizer.transform(data).T 
print(term_doc_mat)

<Compressed Sparse Column sparse matrix of dtype 'int64'
	with 242954 stored elements and shape (4708, 8395)>
  Coords	Values
  (393, 0)	2
  (481, 0)	1
  (533, 0)	1
  (576, 0)	1
  (838, 0)	1
  (939, 0)	1
  (1009, 0)	1
  (1418, 0)	1
  (1503, 0)	1
  (1909, 0)	1
  (1985, 0)	1
  (2016, 0)	1
  (2136, 0)	1
  (2265, 0)	2
  (2556, 0)	1
  (3129, 0)	2
  (4152, 0)	1
  (4157, 0)	3
  (4404, 0)	1
  (4531, 0)	1
  (4538, 0)	1
  (4601, 0)	1
  (4684, 0)	1
  (4687, 0)	2
  (4703, 0)	1
  :	:
  (1395, 8394)	1
  (1726, 8394)	1
  (1861, 8394)	1
  (1976, 8394)	1
  (2074, 8394)	1
  (2268, 8394)	2
  (2739, 8394)	2
  (2840, 8394)	1
  (2886, 8394)	1
  (2903, 8394)	1
  (3238, 8394)	2
  (3529, 8394)	1
  (3686, 8394)	1
  (3794, 8394)	1
  (4058, 8394)	1
  (4157, 8394)	4
  (4180, 8394)	1
  (4190, 8394)	1
  (4233, 8394)	3
  (4298, 8394)	1
  (4328, 8394)	1
  (4402, 8394)	1
  (4538, 8394)	1
  (4558, 8394)	1
  (4601, 8394)	1


In [33]:
term_doc_mat.shape

(4708, 8395)

In [2]:
from gensim.utils import simple_preprocess


ModuleNotFoundError: No module named 'gensim'